# 02 – Adding area data: are slower areas poorer, older or less healthy?

**Question:** In notebook 01 we saw that average Category 2 response times differ a lot between ambulance trusts.
Here we bring in information about the *people* each trust serves, and see whether it lines up with response times.

**This notebook:**
1. Loads the clean ambulance file from notebook 01
2. Loads the OHID Local Authority Health Profiles (Fingertips) data
3. Picks a few useful area measures
4. Links each council area to its ambulance trust (a lookup table we build ourselves)
5. Combines council areas into trust-level figures, weighted by population
6. Compares these with response times, with charts and correlations

**Data source:** Office for Health Improvement and Disparities (OHID), *Local Authority Health Profiles*,
data for Counties & Unitary Authorities (from April 2023), downloaded October 2026. Public, aggregated data.

## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (10, 5)

PROJECT_DIR = Path("..")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIG_DIR = PROJECT_DIR / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load the data

- The **ambulance** file is the clean one we saved in notebook 01 (run that first).
- The **health profiles** file is the raw download. We look for any CSV in `data/raw` whose name starts with `indicators-CountiesUAs`.

In [ ]:
amb = pd.read_csv(PROCESSED_DIR / "ambulance_response_times_clean.csv", parse_dates=["date"])
print("Ambulance rows:", len(amb))

hp_files = sorted(RAW_DIR.glob("indicators-CountiesUAs*.csv"))
assert hp_files, f"No health profiles CSV found in {RAW_DIR.resolve()}"
hp = pd.read_csv(hp_files[-1], low_memory=False)
print("Health profiles rows, columns:", hp.shape)
hp.head(3)

## 3. What is in the health profiles file?

This file is in **long format**: one row per *indicator × area × sex × age × time period*.
Let's list the indicators and their latest time period.

In [ ]:
indicators = (
    hp.groupby(["Indicator ID", "Indicator Name"])["Time period"]
      .agg(latest="max", periods="nunique")
      .reset_index()
)
indicators

In [ ]:
print(hp["Area Type"].value_counts())
print("Number of council areas:", hp.loc[hp["Area Type"] != "England", "Area Code"].nunique())

## 4. Choose the measures we need

We pick measures that could plausibly be linked to ambulance demand or difficulty:

| Measure | Indicator ID | Why it might matter |
|---|---|---|
| **Deprivation score (IMD 2019)** | 93553 | Poorer areas tend to have more illness and more 999 calls |
| **% population aged 65+** | 92310 | Older people need ambulances more often, and older populations are often more rural |
| **Under-75 mortality rate (all causes)** | 108 | A general measure of how healthy the population is |
| **Population** | (the *Denominator* of 92310) | Needed for weighting |

We use **Persons** (everyone, not split by sex) and the **latest time period** for each.

⚠️ **Limitation:** the deprivation score is from IMD **2019**, the version in this file. A newer IMD has since been published,
so this could be updated later.

In [ ]:
areas_only = hp[(hp["Area Type"] != "England") & (hp["Sex"] == "Persons")]

def latest_values(indicator_id, value_col="Value"):
    """Return one value per council area for the latest time period of an indicator."""
    rows = areas_only[areas_only["Indicator ID"] == indicator_id]
    latest_period = rows.loc[rows["Time period Sortable"].idxmax(), "Time period"]
    rows = rows[rows["Time period"] == latest_period]
    print(f"{indicator_id}: {rows['Indicator Name'].iloc[0]} – period {latest_period}, {len(rows)} areas")
    return rows.set_index("Area Code")[value_col]

areas = pd.DataFrame({
    "area_name":       areas_only.drop_duplicates("Area Code").set_index("Area Code")["Area Name"],
    "imd_score":       latest_values(93553),
    "pct_65_plus":     latest_values(92310),
    "population":      latest_values(92310, value_col="Denominator"),
    "u75_mortality":   latest_values(108),
})
areas.head()

In [ ]:
# Check for missing values
areas.isna().sum()

In [ ]:
# Small areas sometimes have missing values (e.g. Isles of Scilly, City of London)
areas[areas.isna().any(axis=1)]

The two missing areas are the **Isles of Scilly** and the **City of London**. Both are tiny (around 2,000 and 9,000 residents),
so values for them are usually combined with neighbouring areas. Leaving them out makes almost no difference to trust totals.

## 5. Link each council area to its ambulance trust

There is no ready-made official lookup from council areas to ambulance trusts, so we build one.
Most trusts follow region boundaries, with a few exceptions, for example:
- **North and North East Lincolnshire** are in the Yorkshire and the Humber *region*, but are covered by **East Midlands** Ambulance Service.
- **Milton Keynes, Buckinghamshire, Oxfordshire, Berkshire and Hampshire** are covered by **South Central**, while Kent, Surrey and Sussex are **South East Coast**.
- The **Isle of Wight** has its own ambulance service.

⚠️ **Limitation:** a few small border areas are served by a neighbouring trust (e.g. part of north-east Hampshire, Glossop in Derbyshire).
We assign whole council areas, so this is a close approximation.

In [ ]:
TRUST_AREAS = {
    "London": "ALL_LONDON_BOROUGHS",
    "North East": [
        "County Durham", "Darlington", "Gateshead", "Hartlepool", "Middlesbrough",
        "Newcastle upon Tyne", "North Tyneside", "Northumberland", "Redcar and Cleveland",
        "South Tyneside", "Stockton-on-Tees", "Sunderland",
    ],
    "North West": [
        "Blackburn with Darwen", "Blackpool", "Bolton", "Bury", "Cheshire East",
        "Cheshire West and Chester", "Cumberland", "Halton", "Knowsley", "Lancashire",
        "Liverpool", "Manchester", "Oldham", "Rochdale", "Salford", "Sefton", "St. Helens",
        "Stockport", "Tameside", "Trafford", "Warrington", "Westmorland and Furness",
        "Wigan", "Wirral",
    ],
    "Yorkshire": [
        "Barnsley", "Bradford", "Calderdale", "Doncaster", "East Riding of Yorkshire",
        "Kingston upon Hull", "Kirklees", "Leeds", "North Yorkshire UA", "Rotherham",
        "Sheffield", "Wakefield", "York",
    ],
    "East Midlands": [
        "Derby", "Derbyshire", "Leicester", "Leicestershire", "Lincolnshire",
        "North East Lincolnshire", "North Lincolnshire", "North Northamptonshire",
        "West Northamptonshire", "Nottingham", "Nottinghamshire", "Rutland",
    ],
    "West Midlands": [
        "Birmingham", "Coventry", "Dudley", "Herefordshire", "Sandwell", "Shropshire",
        "Solihull", "Staffordshire", "Stoke-on-Trent", "Telford and Wrekin", "Walsall",
        "Warwickshire", "Wolverhampton", "Worcestershire",
    ],
    "East of England": [
        "Bedford", "Cambridgeshire", "Central Bedfordshire", "Essex", "Hertfordshire",
        "Luton", "Norfolk", "Peterborough", "Southend-on-Sea", "Suffolk", "Thurrock",
    ],
    "South East Coast": [
        "Brighton and Hove", "East Sussex", "Kent", "Medway", "Surrey", "West Sussex",
    ],
    "South Central": [
        "Bracknell Forest", "Buckinghamshire UA", "Hampshire", "Milton Keynes", "Oxfordshire",
        "Portsmouth", "Reading", "Slough", "Southampton", "West Berkshire",
        "Windsor and Maidenhead", "Wokingham",
    ],
    "Isle of Wight": ["Isle of Wight"],
    "South Western": [
        "Bath and North East Somerset", "Bournemouth, Christchurch and Poole", "Bristol",
        "Cornwall", "Devon", "Dorset", "Gloucestershire", "Isles of Scilly", "North Somerset",
        "Plymouth", "Somerset UA", "South Gloucestershire", "Swindon", "Torbay", "Wiltshire",
    ],
}

name_to_trust = {}
for trust, names in TRUST_AREAS.items():
    if names == "ALL_LONDON_BOROUGHS":
        # London boroughs (and the City of London) all have codes starting E09
        names = areas.loc[areas.index.str.startswith("E09"), "area_name"].tolist()
    for n in names:
        name_to_trust[n] = trust

areas["trust"] = areas["area_name"].map(name_to_trust)
areas["trust"].value_counts()

### Check the lookup
Every council area should be matched to exactly one trust. If a name was misspelled, it would show up here as missing.

In [ ]:
unmatched = areas[areas["trust"].isna()]
print("Unmatched areas:", len(unmatched))
assert unmatched.empty, unmatched

listed_names = [n for n in name_to_trust]
not_found = set(listed_names) - set(areas["area_name"])
print("Names in lookup but not in data:", not_found or "none")
assert not not_found

In [ ]:
lookup = areas[["area_name", "trust"]].reset_index().rename(columns={"Area Code": "area_code"})
lookup.to_csv(PROCESSED_DIR / "council_area_to_ambulance_trust_lookup.csv", index=False)
print("Saved lookup with", len(lookup), "areas")

## 6. Combine council areas into trust-level figures

A trust covers several council areas of very different sizes. Birmingham has over a million people; Rutland has about 40,000.
So we use a **population-weighted average**: bigger areas count for more.

For example: trust IMD score = sum(area IMD score × area population) ÷ sum(area population).

Areas with a missing value are left out of that measure's average.

In [ ]:
def weighted_mean(group, col):
    g = group.dropna(subset=[col, "population"])
    return (g[col] * g["population"]).sum() / g["population"].sum()

trust_area = pd.DataFrame({
    "population":    areas.groupby("trust")["population"].sum(),
    "n_council_areas": areas.groupby("trust").size(),
    "imd_score":     areas.groupby("trust").apply(weighted_mean, col="imd_score"),
    "pct_65_plus":   areas.groupby("trust").apply(weighted_mean, col="pct_65_plus"),
    "u75_mortality": areas.groupby("trust").apply(weighted_mean, col="u75_mortality"),
})
trust_area.round(1).sort_values("imd_score", ascending=False)

In [ ]:
# Sense check: the trust populations should add up to roughly England's population (~57–58 million)
print(f"Total population: {trust_area['population'].sum():,.0f}")

## 7. Response times for each trust (last 12 months)

Same method as notebook 01: total waiting time ÷ total calls (a weighted average).

In [ ]:
latest = amb["date"].max()
last_12 = amb[(amb["level"] == "Trust") & (amb["date"] > latest - pd.DateOffset(months=12))]

resp = last_12.groupby("name")[["c1_total_sec", "c1_calls", "c2_total_sec", "c2_calls"]].sum()
resp["c1_mean_min"] = resp["c1_total_sec"] / resp["c1_calls"] / 60
resp["c2_mean_min"] = resp["c2_total_sec"] / resp["c2_calls"] / 60

# Calls per 1,000 people per year – a measure of how busy each trust is
trust = trust_area.join(resp[["c1_mean_min", "c2_mean_min", "c2_calls"]], how="inner")
trust["c2_calls_per_1000"] = trust["c2_calls"] / trust["population"] * 1000

print("Trusts matched:", len(trust))
trust.round(1).sort_values("c2_mean_min")

## 8. Do slower trusts serve poorer, older or less healthy populations?

We draw a scatter chart for each measure and calculate a **correlation** (from −1 to +1):
- close to **+1**: higher values go with *slower* responses
- close to **−1**: higher values go with *faster* responses
- close to **0**: no clear link

We use **Spearman's** correlation, which compares rankings and is less affected by one unusual trust.

⚠️ **Important:** we only have **11 trusts**, so these results are a first clue, not proof.
With so few points, a correlation can look strong just by chance, and correlation doesn't tell us what *causes* what.

In [ ]:
measures = {
    "imd_score":        "Deprivation score (IMD 2019)\n(higher = more deprived)",
    "pct_65_plus":      "% population aged 65+",
    "u75_mortality":    "Under-75 mortality rate\n(per 100,000)",
    "c2_calls_per_1000": "Category 2 calls per 1,000 people\n(last 12 months)",
}

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
results = []
for ax, (col, label) in zip(axes.flat, measures.items()):
    rho = trust[col].corr(trust["c2_mean_min"], method="spearman")
    results.append({"measure": col, "spearman_rho": round(rho, 2)})
    ax.scatter(trust[col], trust["c2_mean_min"])
    for name, row in trust.iterrows():
        # East of England and South Western have almost the same response time,
        # so put one label below its dot to stop them overlapping
        offset = (4, -10) if name == "East of England" else (4, 3)
        ax.annotate(name, (row[col], row["c2_mean_min"]), fontsize=8,
                    xytext=offset, textcoords="offset points")
    ax.set_xlabel(label)
    ax.set_ylabel("Category 2 mean (minutes)")
    ax.set_title(f"Spearman correlation = {rho:.2f}")

fig.suptitle(f"Category 2 response time vs area characteristics, by ambulance trust\n(12 months to {latest:%B %Y})")
plt.tight_layout()
plt.savefig(FIG_DIR / "02_c2_vs_area_measures.png", dpi=150)
plt.show()

pd.DataFrame(results)

## 9. Save the combined trust table

In [ ]:
out = PROCESSED_DIR / "trust_response_and_area_measures.csv"
trust.reset_index(names="trust").to_csv(out, index=False)
print("Saved", out)

## 10. Summary

**What we found (12 months to September 2026, 11 trusts):**

- **Deprivation – a surprise!** The link goes the *opposite* way to what we expected (Spearman ≈ −0.7).
  The most deprived trust areas (North East, North West, Yorkshire, West Midlands) have the *fastest* Category 2 responses.
  The slowest trusts (East Midlands, East of England, South Western) serve *less* deprived areas.
- **Under-75 mortality** shows the same pattern (≈ −0.55), which makes sense because it is closely linked to deprivation.
- **% aged 65+** and **calls per 1,000 people** show almost no link (close to 0).
  So the slow trusts are *not* simply busier or serving older populations.

**What might explain it?** The fast trusts mostly cover big, dense urban areas in the North and Midlands.
The slow ones (except London) cover large, rural areas where ambulances travel further.
So **rurality and distance** may matter more than deprivation, which we test in notebook 03.
London is an interesting exception: dense, but still slow, so other factors (like hospital handover delays) probably matter too.

**Limitations to mention in interviews:**
- Only 11 trusts, so results are suggestive, not conclusive.
- Correlation is not causation: staffing, hospital handover delays and geography all play a part.
- Trust-level averages hide big differences between areas within a trust.
- The council-area-to-trust lookup is a close approximation at a few borders.
- Deprivation is from IMD 2019.

**Next notebook (03):** add a direct measure of **rurality / population density** to test the rural-distance idea properly.